### This notebook is written for calculating, visualizing  and investigating of SHap values for each model for  a small part of our data



✅ Summary

This section of code does the following for each 224×224 tile:

Extract tile

Visualize tile

Prepare it for model input

Compute SHAP explanations

Visualize SHAP heatmaps

Print the model's prediction

It is essentially an explainability pipeline for tile-based image classification.

In [ ]:
import tensorflow
from tensorflow import keras
print("Keras Version : {}".format(keras.__version__))
import shap

print("SHAP Version : {}".format(shap.__version__))

In [ ]:
shap.initjs()

In [ ]:
### This is for calculating 

import pandas as pd
df_scores = pd.read_excel(r'../../data/raw_data/ROW_DATA_Sonja_add_MV_ECMO_and_ECMO_LPS.xlsx',sheet_name='Total Score',engine='openpyxl')
df_scores_Ct_MV_LP = df_scores[0:19]


this_data = df_scores['PDF/ EXCEL correspodent Data'].values
df_scores

In [38]:
VALIDATION_DIR = ['1st_part','2nd_part','3rd_part']
MODELS         = '../../models/initial_relu/3fold_CV/effnet_aug_3fold_90_rotation/'
SAMPLES        = '../../results/Grad_cam/initial_relu/samples_V_r_a/main_img_'


### The model is defined, loaded and then shap values are calculated for each tile 

In [ ]:
#n script for reading train and validation data set and train the network
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.layers import Conv2D, Flatten, Input, MaxPooling2D, Activation, BatchNormalization, Concatenate, Dropout, Add, Dense,LeakyReLU
from tensorflow.keras.models import Model
import numpy as np
import matplotlib.pyplot as plt
import glob
import cv2
import pandas as pd
from tensorflow.keras.utils import plot_model
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from keras.models import model_from_json
from pathlib import Path
from tensorflow.keras.utils import plot_model
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from matplotlib import pyplot
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ModelCheckpoint
import tensorflow as tf
from tensorflow.keras.optimizers import SGD,RMSprop
from tensorflow.keras.applications import EfficientNetB0
from keras.applications.vgg16 import VGG16
from keras.preprocessing import image
from keras.applications.vgg16 import preprocess_input
from keras.layers import Input, Flatten, Dense, Lambda
from keras.models import Model
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import glob
import cv2
from pathlib import Path
from tensorflow.keras.utils import plot_model
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from keras.models import model_from_json
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score
from matplotlib import pyplot
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import f1_score
from sklearn.metrics import auc
import keras
import tensorflow as tf
from keras.callbacks import ModelCheckpoint
import pandas as pd
import numpy as np
from tensorflow.keras import backend as K
from tensorflow.python.keras.layers import InputSpec, Layer

import pandas as pd
from sklearn.metrics import classification_report
from keras.models import Model, model_from_json

from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import plot_precision_recall_curve
from sklearn.metrics import classification_report, confusion_matrix
%matplotlib inline
from tensorflow.keras.applications import EfficientNetB0,EfficientNetB4,EfficientNetB7
import os
##https://neptune.ai/blog/keras-metrics
from keras.callbacks import Callback
import matplotlib.pyplot as plt
import numpy as np
from scikitplot.metrics import plot_confusion_matrix, plot_roc, plot_precision_recall, plot_precision_recall_curve
import shap
######################################################################################################
class MulticlassTruePositives(tf.keras.metrics.Metric):
    def __init__(self, name='multiclass_true_positives', **kwargs):
        super(MulticlassTruePositives, self).__init__(name=name, **kwargs)
        self.true_positives = self.add_weight(name='tp', initializer='zeros')

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.reshape(tf.argmax(y_pred, axis=1), shape=(-1, 1))
        values = tf.cast(y_true, 'int32') == tf.cast(y_pred, 'int32')
        values = tf.cast(values, 'float32')
        if sample_weight is not None:
            sample_weight = tf.cast(sample_weight, 'float32')
            values = tf.multiply(values, sample_weight)
        self.true_positives.assign_add(tf.reduce_sum(values))

    def result(self):
        return self.true_positives

    def reset_state(self):
        # The state of the metric will be reset at the start of each epoch.
        self.true_positives.assign(0.)

## Model definition
#################################################################################################################################
def pretrained_model_func(img_shape, num_classes,layer_type):
    model_conv = EfficientNetB4(weights='imagenet', include_top=False)

    #Input format
    keras_input = Input(shape=(224,224,3), name = 'image_input')
    
    #Use the generated model 
    output_conv = model_conv(keras_input)
    
    for layer in model_conv.layers:
        layer.trainable = False
    
    #Add the fully-connected layers 
    x = Flatten(name='flatten')(output_conv)
    x = Dense(128, activation=layer_type, name='fc1')(x)
    x = Dense(128, activation=layer_type, name='fc2')(x)
    x = Dropout(0.5, name='dropout')(x)
    x = Dense(num_classes, activation='softmax', name='predictions')(x)
    
    METRICS = [
      keras.metrics.SparseCategoricalAccuracy(),
      MulticlassTruePositives()]
    
    
    #Create your own model 
    pretrained_model = Model(inputs=keras_input, outputs=x)
    pretrained_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=METRICS)
    #pretrained_model.summary()

    return pretrained_model


#######################################################################
###Here we add a column to our dataframe to compare different models

first_exp_col =np.zeros(len(df_scores['(sanned slide) Sample Name ']))
    
#validation_slide = ['ECMO','Control','ECMO+LPS','MV+LPS','MV']
validation_slide = VALIDATION_DIR
model_name       = 'EffnetB4_man_aug'
### I took the best models in terms of TP 

tile_based_scores = {}
image_based_scores = {}
slide_based_scores = {}
for val_s in validation_slide:
    val_dir          = val_s # loop over all groups as validation dataset
    Sample_Slide     = {'Control' : [9, 14, 15, 16, 17, 18, 19, 20, 26, 28, 46, 47],  
                    'MV+LPS'  : [35, 36, 41, 42, 43, 44, 45],
                    'ECMO+LPS': [1, 2 ,4, 5, 6, 7, 8, 25, 37, 38, 48],
                    'MV'      : [3, 10, 13, 28, 34],
                    'ECMO'    : [11, 12, 29, 30, 31, 32, 33, 39, 40, 49]}




    validation_DATA_PATH  = val_dir+'_validation/'
    validation_LABEL_PATH = '../../data/folded_data/3fold_CV/All_samples_total_score_Median/'+val_dir+'_val_total_score/validation/validation_newlabels.txt'

    ### Max and min of Median column 
    max_score = max(df_scores['Median'])
    min_score = min(df_scores['Median'])

    ##Normalized thresholds
    low = (10-(min_score))/(max_score-min_score)
    mid = (25-(min_score))/(max_score-min_score)
    
    ## Initialize validation samples
    input_shape = (224,224,3)


    pretrained_model = pretrained_model_func(input_shape, 3,'relu') ## REGRESSION: number of classes 1 instead of 3
    weights =glob.glob(MODELS+val_dir+'*_best_weights.*')
    print(weights)
    # Load weights, select between best_weights (val_accuracy) and final_weights from last epoch
    pretrained_model.load_weights(weights[0])
    
    

    imgs = glob.glob(SAMPLES+val_dir+'/*.tif')

    print(imgs)

    for im in imgs:
        img_path = im

        if cv2.imread(img_path) is not None: ## Check if the image exists
            input_img = cv2.imread(img_path)
            print('not None')
            plt.imshow(input_img)
            max_tile_i = int(np.floor(input_img.shape[0]/224))  ##maxiumum number of rows
            max_tile_j = int(np.floor(input_img.shape[1]/224))  ##maxiumum number of columns

           
            ### We need to feed the tiles into model to get the activation map
            for i in range(max_tile_i):
                for j in range(max_tile_j):
                    tmp_img = input_img[224*i:224*(i+1),224*j:224*(j+1),:]
                    plt.imshow(tmp_img)
                    #tmp_img = cv2.resize(tmp_img,(224,224))
                    y = np.expand_dims(tmp_img, axis=0)  
                    print(tmp_img.shape)
                    img_array = tmp_img
                    masker = shap.maskers.Image("inpaint_ns",tmp_img.shape)
                    explainer = shap.Explainer(pretrained_model, masker, output_names=['low','medium','high'])
                    shap_values = explainer(y, outputs=shap.Explanation.argsort.flip[:3])
                    plt.figure()
                    shap.image_plot(shap_values)
                    plt.imshow(shap_values.data[0,:,:,:].astype(int))
                    print(pretrained_model.predict(y))
                   
    
    